In [1]:
import pandas as pd
import os
from google.colab import files

# Load Data

In [2]:
file_penjualan = "penjualan barang.csv"
file_pemasukan = "pemasukan barang.csv"

In [5]:
if not os.path.exists(file_penjualan) or not os.path.exists(file_pemasukan):
    print("🛑 ERROR: File sumber tidak ditemukan di Colab!")
    print("Solusi: Silakan klik ikon Folder (📁) di kiri, lalu upload ulang file 'penjualan barang.csv' dan 'pemasukan barang.csv'.")
else:
    try:
        # Membaca Data
        df_penjualan = pd.read_csv(file_penjualan)
        df_pemasukan = pd.read_csv(file_pemasukan)

        # Membersihkan Data (Menghapus kolom indeks bawaan yang mengganggu visualisasi)
        if 'Unnamed: 0' in df_penjualan.columns:
            df_penjualan = df_penjualan.drop(columns=['Unnamed: 0'])
        if 'Unnamed: 0' in df_pemasukan.columns:
            df_pemasukan = df_pemasukan.drop(columns=['Unnamed: 0'])

        # Mengubah format tanggal agar terbaca sempurna sebagai "Date" di Looker Studio
        df_penjualan['tanggal'] = pd.to_datetime(df_penjualan['tanggal'], errors='coerce')
        df_pemasukan['tanggal'] = pd.to_datetime(df_pemasukan['tanggal'], errors='coerce')

        # Membuat Tabel Agregasi Status Stok (Gabungan Masuk & Keluar)
        stok_keluar = df_penjualan.groupby('nama.barang')['kuantum'].sum().reset_index()
        stok_keluar.rename(columns={'kuantum': 'total_terjual'}, inplace=True)

        stok_masuk = df_pemasukan.groupby('nama.barang')['kuantum'].sum().reset_index()
        stok_masuk.rename(columns={'kuantum': 'total_masuk'}, inplace=True)

        # Menggabungkan data (JOIN)
        status_stok = pd.merge(stok_masuk, stok_keluar, on='nama.barang', how='outer').fillna(0)
        status_stok['sisa_stok'] = status_stok['total_masuk'] - status_stok['total_terjual']

        # MENYIMPAN HASIL KE FILE CSV BARU (index=False agar rapi)
        nama_file_1 = "penjualan_bersih_looker.csv"
        nama_file_2 = "status_stok_looker.csv"

        df_penjualan.to_csv(nama_file_1, index=False)
        status_stok.to_csv(nama_file_2, index=False)

        print("✅ Data berhasil dibersihkan dan digabungkan!")

        # MENDOWNLOAD FILE OTOMATIS KE LAPTOP/KOMPUTER
        print(f"📥 Sedang mendownload {nama_file_1}...")
        files.download(nama_file_1)

        print(f"📥 Sedang mendownload {nama_file_2}...")
        files.download(nama_file_2)

        print("🎉 Selesai! Cek folder 'Downloads' di komputer kamu.")

    except Exception as e:
        print(f"🛑 ERROR SAAT MEMPROSES: {e}")

✅ Data berhasil dibersihkan dan digabungkan!
📥 Sedang mendownload penjualan_bersih_looker.csv...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Sedang mendownload status_stok_looker.csv...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🎉 Selesai! Cek folder 'Downloads' di komputer kamu.
